In [1]:
import os
import numpy as np
import tensorflow as tf

In [2]:
print("Loading raw training data for calibration...")
base_cache = np.load("ptbxl_preprocessed_cache.npz", allow_pickle=True)
X_train = base_cache["X_train"]

print("Loading saved 1D-CNN model weights...")
cnn_1d = tf.keras.models.load_model("tinyml_1d_cnn.keras")

def representative_data_gen():
    # Use 200 calibration windows from training set to determine INT8 scaling
    for i in range(200):
        yield [X_train[i:i+1].astype(np.float32)]

print("Converting and Quantizing model to INT8...")
converter = tf.lite.TFLiteConverter.from_keras_model(cnn_1d)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Force strict integer precision
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_path = "esp32_ecg_model_quant.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_quant_model)

raw_size = os.path.getsize(tflite_path) / 1024.0
print(f"Quantized Model Size: {raw_size:.2f} KB")

# Generate the C-array header for ESP-IDF/Arduino
header_path = "model_data.h"
with open(header_path, "w") as f:
    f.write(f"// Auto-generated TFLM Model Header\n")
    f.write(f"#ifndef ESP32_ECG_MODEL_H_\n#define ESP32_ECG_MODEL_H_\n\n")
    f.write(f"const unsigned int g_model_len = {len(tflite_quant_model)};\n")
    f.write(f"const unsigned char g_model[] __attribute__((aligned(4))) = {{\n  ")
    for i, b in enumerate(tflite_quant_model):
        f.write(f"0x{b:02x}, ")
        if (i + 1) % 12 == 0:
            f.write("\n  ")
    f.write("\n};\n\n#endif  // ESP32_ECG_MODEL_H_\n")

print(f"Generated embedded C-header file: '{header_path}'")

Loading raw training data for calibration...
Loading saved 1D-CNN model weights...
Converting and Quantizing model to INT8...
INFO:tensorflow:Assets written to: C:\Users\Asus\AppData\Local\Temp\tmp14bw_m2a\assets


INFO:tensorflow:Assets written to: C:\Users\Asus\AppData\Local\Temp\tmp14bw_m2a\assets


Saved artifact at 'C:\Users\Asus\AppData\Local\Temp\tmp14bw_m2a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1000, 3), dtype=tf.float32, name='ecg_input')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1356652767824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652768784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652768208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652768592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652767440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652768400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652769168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652767632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652770128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652769936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1356652769744

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Quantized Model Size: 24.95 KB
Generated embedded C-header file: 'model_data.h'
